**1 implementation option**

In [1]:
from pyspark.sql import SparkSession, Row
import numpy as np
import random

from pyspark.ml.param.shared import Param, Params, HasMaxIter, HasStepSize
from pyspark.ml import Estimator, Model
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

np.random.seed(42)

In [3]:
# Spark session
spark = SparkSession.builder.appName("LinearRegressionGD").getOrCreate()
sc = spark.sparkContext

In [4]:
# Hidden weights
true_weights = np.array([1.5, 0.3, -0.7])

# Random features
def generate_point(_):
    x = np.random.randn(3)
    y = float(np.dot(true_weights, x) + np.random.normal(scale=0.1))
    return (x.tolist(), y)

rdd = sc.parallelize(range(100000)).map(generate_point)

In [5]:
# Class Params
class LinearRegressionParams(HasMaxIter, HasStepSize):
    def __init__(self):
        super().__init__()
        self._setDefault(maxIter=100, stepSize=0.01)

In [6]:
# Class Estimator
class LinearRegression(Estimator, LinearRegressionParams):
    def __init__(self):
        super().__init__()
        
    def setMaxIter(self, value):
        return self._set(maxIter=value)

    def setStepSize(self, value):
        return self._set(stepSize=value)

    def _fit(self, dataset):
        rdd = dataset.rdd.map(lambda row: (np.array(row.features), row.label))
        weights = np.zeros(rdd.first()[0].shape)
        count = rdd.count()

        for _ in range(self.getOrDefault(self.maxIter)):
            gradient = rdd.map(lambda x: (x[0], x[1] - np.dot(weights, x[0]))) \
                          .map(lambda x: -2 * x[0] * x[1]) \
                          .reduce(lambda a, b: a + b) / count
            
            weights -= self.getOrDefault(self.stepSize) * gradient

        return LinearRegressionModel(weights)

In [7]:
# Class Model
class LinearRegressionModel(Model, LinearRegressionParams):
    def __init__(self, weights):
        super().__init__()
        self.weights = weights

    def _transform(self, dataset):
        predict_udf = udf(lambda x: float(np.dot(self.weights, x)), DoubleType())
        return dataset.withColumn("prediction", predict_udf("features"))

In [9]:
df = rdd.map(lambda x: Row(features=Vectors.dense(x[0]), label=x[1])).toDF()

In [10]:
# Model training
lr = LinearRegression().setMaxIter(500).setStepSize(0.01)
model = lr.fit(df)

predictions = model._transform(df)
predictions.show(5)

+--------------------+-------------------+-------------------+
|            features|              label|         prediction|
+--------------------+-------------------+-------------------+
|[1.04786578950256...|   1.56582123867901| 1.4495936863876746|
|[-0.9786274002353...|-1.6951499075930407|-1.7337530900108733|
|[-1.0084207910607...|-0.6931277600571741|-0.5837704902305756|
|[-0.9994758088636...|-1.9506523937117248| -2.104678936765348|
|[-0.6925976907028...|-1.4871958714552997|-1.4214976674626008|
+--------------------+-------------------+-------------------+
only showing top 5 rows



In [11]:
# Checking the scales
print("Learned weights:", model.weights)
print("True weights:   ", true_weights)

Learned weights: [ 1.49987204  0.30017359 -0.70008224]
True weights:    [ 1.5  0.3 -0.7]


------------------------------------------------------------------------------------------------------------

**2 implementation option**

In [1]:
from pyspark.sql import SparkSession, Row
from pyspark.ml import Estimator, Model, Pipeline
from pyspark.ml.param.shared import Param, Params, HasMaxIter, HasStepSize
from pyspark.ml.feature import (
    SQLTransformer, StringIndexer, OneHotEncoder, Imputer,
    VectorAssembler, StandardScaler, ChiSqSelector
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
from pyspark.ml.linalg import Vectors, VectorUDT
import numpy as np
import random
import unittest


np.random.seed(42)

In [3]:
# Spark session
spark = SparkSession.builder.appName("CustomLinearRegressionPipeline").getOrCreate()
sc = spark.sparkContext

In [5]:
# Data Generation 
def generate_data(i):
    category = random.choice(["A", "B", "C"])
    numeric1 = np.random.randn()
    numeric2 = np.random.randn() if random.random() > 0.1 else None  # introduce missing
    label = 2.5 * numeric1 + (-1.3 if category == "A" else 0.8 if category == "B" else 0.0)
    label += np.random.normal(scale=0.1)
    return Row(category=category, numeric1=float(numeric1), numeric2=numeric2, label=float(label))

rdd = sc.parallelize(range(100000)).map(generate_data)
df = spark.createDataFrame(rdd)

In [6]:
# Class Params
class LinearRegressionParams(HasMaxIter, HasStepSize):
    def __init__(self):
        super().__init__()
        self._setDefault(maxIter=100, stepSize=0.01)

In [7]:
# Class Estimator
class LinearRegression(Estimator, LinearRegressionParams):
    def __init__(self):
        super().__init__()
        
    def setMaxIter(self, value):
        return self._set(maxIter=value)

    def setStepSize(self, value):
        return self._set(stepSize=value)

    def _fit(self, dataset):
        rdd = dataset.rdd.map(lambda row: (np.array(row.features), row.label))
        weights = np.zeros(rdd.first()[0].shape)
        count = rdd.count()

        for _ in range(self.getOrDefault(self.maxIter)):
            gradient = rdd.map(lambda x: (x[0], x[1] - np.dot(weights, x[0]))) \
                          .map(lambda x: -2 * x[0] * x[1]) \
                          .reduce(lambda a, b: a + b) / count
            
            weights -= self.getOrDefault(self.stepSize) * gradient

        return LinearRegressionModel(weights)

In [8]:
# Class Model
class LinearRegressionModel(Model, LinearRegressionParams):
    def __init__(self, weights):
        super().__init__()
        self.weights = weights

    def _transform(self, dataset):
        predict_udf = udf(lambda x: float(np.dot(self.weights, x)), DoubleType())
        return dataset.withColumn("prediction", predict_udf("features"))

In [9]:
# Pipeline
pipeline = Pipeline(stages=[
    SQLTransformer(statement="SELECT *, numeric1 * 2 as numeric1_double FROM __THIS__"),
    StringIndexer(inputCol="category", outputCol="category_index"),
    OneHotEncoder(inputCol="category_index", outputCol="category_vec", dropLast=False),
    Imputer(inputCols=["numeric2"], outputCols=["numeric2_imputed"]),
    VectorAssembler(
        inputCols=["numeric1", "numeric1_double", "numeric2_imputed", "category_vec"],
        outputCol="assembled"
    ),
    StandardScaler(inputCol="assembled", outputCol="scaled"),
    SQLTransformer(statement="SELECT *, scaled as features FROM __THIS__"),
    LinearRegression().setMaxIter(500).setStepSize(0.01)
])

In [10]:
# Model training
model = pipeline.fit(df)
predictions = model._transform(df)
predictions.show(5)

+--------+--------------------+--------------------+-------------------+--------------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|category|            numeric1|            numeric2|              label|     numeric1_double|category_index| category_vec|    numeric2_imputed|           assembled|              scaled|            features|         prediction|
+--------+--------------------+--------------------+-------------------+--------------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|       A| 0.15217248190065263|  0.7050645237022386|-1.0925044881478891| 0.30434496380130527|           1.0|(3,[1],[1.0])|  0.7050645237022386|[0.15217248190065...|[0.15334468619361...|[0.15334468619361...|-0.9195485057718826|
|       C|  -1.431228970145038| -0.6240039466334272| -3.538234198620681|  -2.862457940290076

In [11]:
# Quality metrics
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
print("RMSE:", evaluator_rmse.evaluate(predictions))
print("R2:  ", evaluator_r2.evaluate(predictions))

RMSE: 0.09934927335278167
R2:   0.9985719774516791


-----------------------------------------------------------------------------------------------------------------

In [12]:
# Unit Tests
class TestLinearRegression(unittest.TestCase):
    def test_pipeline_runs(self):
        try:
            model = pipeline.fit(df)
            result = model.transform(df)
            self.assertTrue("prediction" in result.columns)
        except Exception as e:
            self.fail(f"Pipeline failed with exception: {e}")

    def test_model_accuracy(self):
        model = pipeline.fit(df)
        preds = model.transform(df)
        rmse = evaluator_rmse.evaluate(preds)
        self.assertLess(rmse, 1.0, "RMSE is too high, model not learning")

unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestLinearRegression))

.                                                                               
----------------------------------------------------------------------
Ran 2 tests in 689.031s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>